## Read data

demonstration data set from the UCR collection

In [1]:
import numpy as np

data = [np.load(f'../data/GestureMidAirD1/{variable}_{set_name}.npy')
        for variable in ['X', 'y'] for set_name in ['train', 'test']]

X_train, X_test, y_train, y_test = data

print("X_train dims: ", X_train.shape)
print("X_test dims: ", X_test.shape)

X_train dims:  (208, 1, 360)
X_test dims:  (130, 1, 360)


in order to apply the foundation model to your data, `X_train` and `X_test` should be of the shape `(n_samples, n_channels=1, seq_len)`, where `seq_len` is a multiple of 32

if original sequence length is different, resize it, for example, using the following function:

In [2]:
import torch
import torch.nn.functional as F

def resize(X):
    X_scaled = F.interpolate(torch.tensor(X, dtype=torch.float), size=512, mode='linear', align_corners=False)
    return X_scaled.numpy()
    
X_train, X_test = resize(X_train), resize(X_test)

print("X_train dims: ", X_train.shape)
print("X_test dims: ", X_test.shape)

X_train dims:  (208, 1, 512)
X_test dims:  (130, 1, 512)


## Evaluation Protocol

In [3]:
from mantis.architecture import MantisV1, MantisV2, Mantis8M
from mantis.trainer import MantisTrainer
from sklearn.ensemble import RandomForestClassifier
    
device = 'cpu' # set device

def evalute_model(network, layer_idx):
    model = MantisTrainer(device=device, network=network) # init trainer
    Z_train = model.transform(X_train)
    Z_test = model.transform(X_test)

    predictor = RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=0)
    predictor.fit(Z_train, y_train)
    
    y_pred = predictor.predict(Z_test)
    print(f'Accuracy at the layer {layer_idx}: {np.mean(y_test == y_pred)}')

/Users/vfeofanov/Library/Caches/pypoetry/virtualenvs/mantis-tsfm-uAw64xg6-py3.11/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Results

We are going to test the performance of each model layer by layer, i.e., trying each transformer layer's output as the final embedding.

### Original Mantis Checkpoint

By default, the classification token (`output_token='cls_token'`) is used as the model's output.

In [21]:
for layer_idx in range(6):
    network = MantisV1(device=device, return_transf_layer=layer_idx) # init model
    network = network.from_pretrained("paris-noah/Mantis-8M") # load weights
    evalute_model(network, layer_idx)

Accuracy at the layer 0: 0.6538461538461539
Accuracy at the layer 1: 0.676923076923077
Accuracy at the layer 2: 0.6538461538461539
Accuracy at the layer 3: 0.6692307692307692
Accuracy at the layer 4: 0.6615384615384615
Accuracy at the layer 5: 0.6692307692307692


As we can see, one of the intermediate layers (in this case, layer 1) may outperform the last transformer layer.

Now let's look at the performance when the concatenation of the cls token and the mean token over non-classification tokens is used as the model's output.

In [31]:
for layer_idx in range(6):
    network = MantisV1(device=device, return_transf_layer=layer_idx, output_token='combined') # init model
    network = network.from_pretrained("paris-noah/Mantis-8M") # load weights
    evalute_model(network, layer_idx)

Accuracy at the layer 0: 0.676923076923077
Accuracy at the layer 1: 0.7
Accuracy at the layer 2: 0.7
Accuracy at the layer 3: 0.7153846153846154
Accuracy at the layer 4: 0.6846153846153846
Accuracy at the layer 5: 0.6923076923076923


The performance is even further improved.

### Mantis Plus Checkpoint

In [25]:
for layer_idx in range(6):
    network = MantisV1(device=device, return_transf_layer=layer_idx) # init model
    network = network.from_pretrained("paris-noah/MantisPlus") # load weights
    evalute_model(network, layer_idx)

Accuracy at the layer 0: 0.6538461538461539
Accuracy at the layer 1: 0.6923076923076923
Accuracy at the layer 2: 0.6307692307692307
Accuracy at the layer 3: 0.6692307692307692
Accuracy at the layer 4: 0.676923076923077
Accuracy at the layer 5: 0.6692307692307692


In [26]:
for layer_idx in range(6):
    network = MantisV1(device=device, return_transf_layer=layer_idx, output_token='combined') # init model
    network = network.from_pretrained("paris-noah/MantisPlus") # load weights
    network.return_transf_layer = layer_idx
    network.output_token = 'combined'
    evalute_model(network, layer_idx)

Accuracy at the layer 0: 0.6692307692307692
Accuracy at the layer 1: 0.6923076923076923
Accuracy at the layer 2: 0.6538461538461539
Accuracy at the layer 3: 0.7
Accuracy at the layer 4: 0.6615384615384615
Accuracy at the layer 5: 0.7


### MantisV2

In [28]:
for layer_idx in range(6):
    network = MantisV2(device=device, return_transf_layer=layer_idx) # init model
    network = network.from_pretrained("paris-noah/MantisV2") # load weights
    evalute_model(network, layer_idx)

Accuracy at the layer 0: 0.676923076923077
Accuracy at the layer 1: 0.676923076923077
Accuracy at the layer 2: 0.7153846153846154
Accuracy at the layer 3: 0.7
Accuracy at the layer 4: 0.7076923076923077
Accuracy at the layer 5: 0.6846153846153846


In [30]:
for layer_idx in range(6):
    network = MantisV2(device=device, return_transf_layer=layer_idx, output_token='combined') # init model
    network = network.from_pretrained("paris-noah/MantisV2") # load weights
    evalute_model(network, layer_idx)

Accuracy at the layer 0: 0.6384615384615384
Accuracy at the layer 1: 0.7
Accuracy at the layer 2: 0.7307692307692307
Accuracy at the layer 3: 0.7
Accuracy at the layer 4: 0.7307692307692307
Accuracy at the layer 5: 0.6615384615384615
